# ClemAgent class definitions

This notebook defines all available agent classes. Import into an experiment notebook with:
```python
%run ./agents.ipynb
```
The calling notebook must have `MODEL` and `GAME` defined before any agent is instantiated.

In [ ]:
import os, json, re
from pathlib import Path
from openai import OpenAI
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model

GAME_TAGS = json.loads(Path("game_tags.json").read_text())

In [ ]:
class BaselineAgentPlayer(ClemAgent):
    """
    Simple example agent.

    It calls the model with the current interaction history and
    uses the model's response as the next guess.
    """

    def __init__(self):
        super().__init__()
        self.model = load_model(MODEL, gen_args=dict(temperature=0.0, max_tokens=None))

    def act(self, last: ClemObservation) -> str:
        # Use the full history (which usually already includes the last observation)
        _, _, response_text = self.model.generate_response(self.history)
        print(self.history)
        # Observe own response in the interaction history
        self.observe(dict(role="user", content=response_text))
        return response_text

In [ ]:
class BasePlanningAgent(ClemAgent):
    """
    Simple example agent with an additional planning loop
    Asks the model to reason about the next move and extracts the answer after the ACT: tag.
    """

    def __init__(self):
        super().__init__()
        self.model = load_model(MODEL, gen_args=dict(temperature=0.0, max_tokens=300))

    def act(self, last: ClemObservation) -> str:

      self.trace = []                     # for transcripts
      demand_prompt = self.history + [
          {"role": "user", "content": "You are a skilled planner. Above you see the game history. What would be the best next action? Openly reason about the next step, then produce your next action exactly as the game expects it preceded by the separator 'ACT:' on its own line. If multiple actions are required, write ACT: first and then all actions."}
      ]
      _, _, act_response = self.model.generate_response(demand_prompt)
      self.trace.append({"type": "planning prompt", "content": json.dumps(demand_prompt, indent=2)})
      self.trace.append({"type": "planning module", "content": act_response})
       

      print("DEBUG: planned response:", act_response)
      if "ACT:" in act_response:
          act_response = act_response.split("ACT:", 1)[1].strip()

      self.observe(dict(role="user", content=act_response))
      self.trace.append({"type": "response", "content": act_response})
        
      print("DEBUG: HISTORY")
      print(self.history)
        
      return act_response

In [ ]:
class FormatCheckingAgent(ClemAgent):
    #no tags for dond since the players exchange messages without tags during all the rounds except the last one

    """
    Simple agent with an additional format checking loop.
    The default version: uses a pre-written tags set from game_tags.json
    """
    def __init__(self):
          super().__init__()
          self.model = load_model(MODEL, gen_args=dict(temperature=0.0, max_tokens=300))
          self.format_requirement = ", ".join(GAME_TAGS.get(GAME, []))
          self.max_retries = 3

    def is_valid(self, response: str) -> bool:
          tags = GAME_TAGS.get(GAME, [])
          if not tags:
              return True
              self.trace.append({"type": f"attempt {attempt + 1} (invalid)", "content": response_text})
          return any(tag in response for tag in tags)
        

    def act(self, last: ClemObservation) -> str:
          self.trace = []
          prompt = self.history.copy()

          for attempt in range(self.max_retries):   
              _, _, response_text = self.model.generate_response(prompt)

              if self.is_valid(response_text):
                  self.trace.append({"type": "attempt","content": f"[attempt {attempt + 1} – valid]\n{response_text}"})
                  return response_text

              # if not valid, reprompt with format reminder
              self.trace.append({"type": f"attempt {attempt + 1} (invalid)", "content": response_text})
              prompt = prompt + [
                  {"role": "assistant", "content": response_text},
                  {"role": "user", "content": f"Invalid format. You must follow this format exactly:{self.format_requirement}"} ]
              print("Player reprompted")

          # return last attempt
          self.trace.append({"type": f"attempt {self.max_retries} (final fallback)", "content": response_text})
          return response_text

In [ ]:
JUDGE_CLIENT = OpenAI(
      api_key="",
      base_url="https://openrouter.ai/api/v1"
  )
JUDGE_MODEL_NAME = "openai/gpt-oss-120b:free"

class FormatCheckingAgent_LLM_judge(ClemAgent):

    """
    Simple agent with an additional format checking loop.
    The non-default version: uses an LLM judge to extract tags at the start of the episode
    """
    def __init__(self):
          super().__init__()
          self.model = load_model(MODEL, gen_args=dict(temperature=0.0, max_tokens=300))
          self.format_requirement = None
          self.max_retries = 3
          self.tags = None

    def get_tags(self, initial_prompt) -> list:
        result = JUDGE_CLIENT.chat.completions.create(
        model=JUDGE_MODEL_NAME,
        temperature=0.0,
        messages=[{"role": "user", "content": f"""You are given a small piece of text which contains gameplay rules.
  You need to extract the necessary tags
  (often written in CAPITAL LETTERS), so that the player can use them for the answer.
  Do not output any text apart from the tag(s). Example IO pair:

  INPUT:
  Let's play a guessing game! You must reply using the format below:
  ANSWER: <some text>

  OUTPUT:
  ["ANSWER:"]

  If you identified no tags, return []. If you identified several tags, return them all in a list.
  Return only the JSON list, no explanation.

  INPUT:
  {initial_prompt}

  OUTPUT:"""}])
        raw = result.choices[0].message.content.strip()
        raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw)
        print(raw)
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return []
        

    def is_valid(self, response: str) -> bool:

          if self.tags is None:
              self.tags = self.get_tags(initial_prompt=self.history[0]["content"])
          if not self.tags:       
              return True
          return any(tag in response for tag in self.tags)  

        

    def act(self, last: ClemObservation) -> str:
          self.tags = self.get_tags(initial_prompt=last.content)
          if self.format_requirement is None:
              self.format_requirement = ", ".join(self.tags) if self.tags else ""
          self.trace = []
          self.trace.append({"type": "tags extracted", "content": str(self.tags)}) 
          prompt = self.history.copy()

          for attempt in range(self.max_retries):   
              _, _, response_text = self.model.generate_response(prompt)

              if self.is_valid(response_text):
                  self.trace.append({"type": "attempt","content": f"[attempt {attempt + 1} – valid]\n{response_text}"})
                  return response_text

              # if not valid, reprompt with format reminder
              self.trace.append({"type": f"attempt {attempt + 1} (invalid)", "content": response_text})
              prompt = prompt + [
                  {"role": "assistant", "content": response_text},
                  {"role": "user", "content": f"Invalid format. You must follow this format exactly:{self.format_requirement}"} ]
              print("Player reprompted")

          # return last attempt
          self.trace.append({"type": f"attempt {self.max_retries} (final fallback)", "content": response_text})
          return response_text

In [ ]:
GAME_STRATEGIES = json.loads(Path("game_strategies.json").read_text())

class UpfrontStrategyAgent(ClemAgent):

    def __init__(self):
        super().__init__()
        self.model = load_model(MODEL, gen_args=dict(temperature=0.0, max_tokens=300))
        self.strategy = None

    def reset(self):
        super().reset()
        self.strategy = None

    def act(self, last: ClemObservation) -> str:
        self.trace = []

        if self.strategy is None:
            self.strategy = GAME_STRATEGIES.get(GAME, "")
            self.trace.append({"type": "strategy", "content": self.strategy})

        prompt = []
        for msg in self.history:
            prompt.append(msg)
            if msg["role"] == "user":
                prompt.append({"role": "assistant", "content": f"[Strategy: {self.strategy}]"})
        self.trace.append({"type": "strategy", "content": self.strategy})

        _, _, response_text = self.model.generate_response(prompt)
        return response_text

In [ ]:
JUDGE_CLIENT = OpenAI(
    api_key="",
    base_url="https://openrouter.ai/api/v1"
)
JUDGE_MODEL_NAME = "deepseek/deepseek-v3.2"

OMEGA = 3
class ReflexionAgent(ClemAgent):

    def __init__(self):
        super().__init__()
        self.agent_id = None
        self.memory = []
        self.model = load_model(MODEL, gen_args=dict(temperature=0.0, max_tokens=300))

    def _memory_path(self):
        return f"reflexion_memory_{GAME}_{self.agent_id}.json"

    def _load_memory(self):
        path = self._memory_path()
        if os.path.exists(path):
            with open(path) as f:
                return json.load(f)
        return []

    def _save_memory(self):
        with open(self._memory_path(), "w") as f:
            json.dump(self.memory, f, indent=2)

    def act(self, last):
        self.trace = []
        prompt = list(self.history)

        if self.memory and len(self.history) == 1:
            memory_text = "Reflections from previous episodes:\n" + "\n".join(f"-\n {m}" for m in self.memory)
            prompt[0] = {**prompt[0], "content": memory_text + "\n\n" + prompt[0]["content"]}
            self.trace.append({"type": "memory loaded", "content": memory_text})

        _, _, response = self.model.generate_response(prompt)
        self.trace.append({"type": "attempt", "content": f"[attempt 1 – valid]\n{response}"})
        return response

    def reflect(self):
        trajectory = "\n".join(
            f"[{m['role']}]: {m['content']}" for m in self.history
        )
        reflection = JUDGE_CLIENT.chat.completions.create(
            model=JUDGE_MODEL_NAME, temperature=0.0, max_tokens=300,
            messages=[{"role": "user", "content":
                f"You just played {GAME} as {self.agent_id}. The episode is over; the next episode will happen shortly.\n"
                f"Trajectory:\n{trajectory}\n\n"
                f"Based on this trajectory, reflect on what happened. "
                f"Did you succeed or fail? What actions led to this result? "
                f"What would you do differently or the same next time? "
                f"Be specific, short and concise."}]
        ).choices[0].message.content

        self.memory.append(reflection)
        if len(self.memory) > OMEGA:
            self.memory.pop(0)
        self._save_memory()
        self.trace.append({"type": "reflection", "content": reflection})